# Domain-Specific RAG Chatbot — Core Notebook
This notebook demonstrates PDF extraction, chunking, pretrained embeddings, FAISS retrieval and source display. The Streamlit app is in `app.py`.

In [ ]:
!pip -q install pypdf sentence-transformers faiss-cpu numpy


In [ ]:
import io, re, numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss


In [ ]:
def extract_pdf_pages(pdf_bytes, source_name='uploaded.pdf'):
    reader=PdfReader(io.BytesIO(pdf_bytes)); pages=[]
    for page_num,page in enumerate(reader.pages,1):
        text=(page.extract_text() or '').strip()
        if text: pages.append({'source':source_name,'page':page_num,'text':text})
    return pages

def make_chunks(pages, size=900, overlap=140):
    chunks=[]
    for page in pages:
        text=re.sub(r'\s+',' ',page['text']).strip(); start=0
        while start<len(text):
            end=min(start+size,len(text)); chunk=text[start:end].strip()
            if chunk: chunks.append({**page,'text':chunk})
            if end==len(text): break
            start=max(0,end-overlap)
    return chunks


In [ ]:
# Upload a PDF in Colab
from google.colab import files
up=files.upload()
pdf_name=next(iter(up))
pages=extract_pdf_pages(up[pdf_name],pdf_name)
chunks=make_chunks(pages)
print('Pages with text:',len(pages),'Chunks:',len(chunks))
print(chunks[0] if chunks else 'No selectable text found')


In [ ]:
# Embed chunks and create FAISS cosine-similarity index
embedder=SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
emb=np.asarray(embedder.encode([c['text'] for c in chunks],normalize_embeddings=True),dtype='float32')
index=faiss.IndexFlatIP(emb.shape[1])
index.add(emb)
print('FAISS vectors:',index.ntotal)


In [ ]:
def ask_document(question,k=4):
    q=np.asarray(embedder.encode([question],normalize_embeddings=True),dtype='float32')
    scores,ids=index.search(q,min(k,len(chunks)))
    results=[(float(s),chunks[int(i)]) for s,i in zip(scores[0],ids[0]) if i>=0]
    if not results or results[0][0]<0.22:
        print('I could not find this information in the uploaded documents.')
        return results
    print('Relevant passages (evidence only):\n')
    for score,item in results:
        print(f"[{item['source']} — page {item['page']}; similarity {score:.3f}]\n{item['text']}\n")
    return results

# Replace with a question answered by your uploaded PDF
# hits=ask_document('What is the main topic of this document?')
